# BÀI TẬP LỚN 02: PHÂN LOẠI CẢM XÚC ĐÁNH GIÁ KHÁCH HÀNG BẰNG MẠNG 5 TẦNG MLP
**Học phần:** Phát triển hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

**Mục tiêu bài thực hành:**
1. Tiền xử lý văn bản đánh giá sản phẩm (Reviews) và mã hóa vector TF-IDF 3003 chiều.
2. Xây dựng mô hình phân loại đa lớp 5 tầng (3003 -> 128 -> 64 -> 32 -> 16 -> 5).
3. Huấn luyện bằng hàm mất mát CrossEntropyLoss.
4. Đánh giá Macro Precision, Macro Recall, Macro F1 và trực quan hóa ma trận nhầm lẫn 5 sao.


In [ ]:
# Cell [1]: Nap du lieu van ban va Trich xuat dac trung TF-IDF
import os, sys, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

candidate_paths = [
    '../data/customer_reviews.csv',
    'data/customer_reviews.csv',
    '../../data/customer_reviews.csv'
]
data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

df_c = pd.read_csv(data_path)
print(f"[INFO] Nap du lieu Customer Reviews: {df_c.shape}")

tfidf = TfidfVectorizer(max_features=3003, stop_words='english')
X_tfidf = tfidf.fit_transform(df_c['review_text'].fillna('')).toarray()
y_label = df_c['rating'].values - 1 # Chuyen ve chi so 0-4

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y_label, test_size=0.2, random_state=42, stratify=y_label)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")


In [ ]:
# Cell [2]: Xay dung mang phan loai da lop 5 tang MLP (3003 -> 5)
class CustomerSentimentMLP(nn.Module):
    def __init__(self, vocab_size=3003, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vocab_size, 128), nn.ReLU(),
            nn.Linear(128, 64),          nn.ReLU(),
            nn.Linear(64, 32),           nn.ReLU(),
            nn.Linear(32, 16),           nn.ReLU(),
            nn.Linear(16, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model_c = CustomerSentimentMLP(vocab_size=X_train.shape[1], num_classes=5)
print(model_c)


In [ ]:
# Cell [3]: Huan luyen mang phan loai da lop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_c.parameters(), lr=0.002)

X_tr_t = torch.FloatTensor(X_train)
y_tr_t = torch.LongTensor(y_train)
X_te_t = torch.FloatTensor(X_test)

for epoch in range(1, 21):
    model_c.train()
    optimizer.zero_grad()
    logits = model_c(X_tr_t)
    loss = criterion(logits, y_tr_t)
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/20] - CrossEntropy Loss: {loss.item():.4f}")

print("[OK] Huan luyen da lop hoan tat!")


In [ ]:
# Cell [4]: Danh gia phan loai da lop va Truc quan hoa Ma tran nham lan
model_c.eval()
with torch.no_grad():
    test_logits = model_c(X_te_t)
    preds = torch.argmax(test_logits, dim=-1).numpy()

acc = accuracy_score(y_test, preds)
print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, preds, target_names=['1 Star', '2 Stars', '3 Stars', '4 Stars', '5 Stars']))

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=[1, 2, 3, 4, 5], yticklabels=[1, 2, 3, 4, 5])
plt.title('Customer Reviews Sentiment Confusion Matrix')
plt.xlabel('Predicted Rating')
plt.ylabel('True Rating')
plt.tight_layout()
plt.show()
